In [1]:
import pandas as pd
import os
import json

from archspaces.core import ArchSpaceCore

Phase 1

In [3]:
# Initialize coordinator

core = ArchSpaceCore()

# Create sample data
df = pd.DataFrame({'R': [0.1, 0.5, 0.9], 'U': [0.2, 0.6, 0.8]})

# Verify DataProcessor delegation
labels = {'R': ['fast', 'avg', 'slow'],'U': ['low', 'avg', 'high']}
discrete_df, tradeoffs = core.discretize(df, n_bins=3, all_labels=labels)
print("Tradeoffs discovered:", tradeoffs)

# Verify RobustnessAnalyzer delegation
robustness, label = core.compute_robustness(discrete_df)
print(f"Robustness: {robustness} for tradeoff: {label}")

Tradeoffs discovered: Counter({'fast,low': 1, 'avg,avg': 1, 'slow,high': 1})
Robustness: 0.3333333333333333 for tradeoff: fast,low


Phase 2

In [7]:
# Verify Declarative Loading
json_path = 'tests/test_system.json'

# Re-create if missing
if not os.path.exists(json_path):
    
    analysis_def = {
        'N_A': [10, 20], 
        'R0': [0.1, 0.2], 
        'U0': [0.5, 0.6], 
        'config': ['c1', 'c2']
    }
    df = pd.DataFrame(analysis_def)
    
    df.to_csv('tests/test_data.csv', index=False)
    
    system_def = {
        "system": {
            "name": "TestSys", 
            "components": {
                "comp1": {
                    "name": "Patt1", 
                    "parameters": 
                    {
                        "N_A": { "type": "integer" }
                    }
                }    
            }
        }, 
        "dataspace": {
            "source_file": "test_data.csv", 
            "quality_objectives": [{"name": "R0", "metric": "response_time"}], 
            "policy_identification": {
                "from": "column", 
                "column": "config", 
                "policies": {
                    "c1": {"name": "Config 1"}, 
                    "c2": {"name": "Config 2"}
                }
            }
        }
    }
    
    with open(json_path, 'w') as f: 
        json.dump(system_def, f)
        
    raw_df, exps_df, outs_df = core.load_detailed_data(json_path)
    print("Experiments Columns:", exps_df.columns.tolist())
    print("Outcomes Columns:", outs_df.columns.tolist())
    
    # Verify Strategy Management (Explanations)
    explanation = core.explain({'robustness': 0.9}, method='template')
    print("Explanation Text:", explanation['text'])

Experiments Columns: ['N_A', 'config']
Outcomes Columns: ['R0']
Explanation Text: Analysis summary:
 - robustness: 0.9


Phase 3

In [2]:
from adept.core.models import Parameter, ParameterLevel, ParameterType
from adept.core.parameter_registry import ParameterRegistry
from adept.core.loader import GenericDataLoader


In [10]:

# 1. Verify Parameter Registry
registry = ParameterRegistry()
p = Parameter(name="replicas", level=ParameterLevel.SYSTEM, type=ParameterType.LEVER, value=5)
registry.register(p)
print(f"Registered parameter: {registry.get('replicas').name}, Type: {registry.get('replicas').type}")
 
# 2. Verify Adaptive Loading
# Create a temporary system.json with adaptive process
test_json = 'manual_test_system.json'
test_csv = 'manual_test_data.csv'
pd.DataFrame({'x': [1, 2], 'y': [0.1, 0.2]}).to_csv(test_csv, index=False)
     
system_def = {
    "system": {
        "name": "ManualTest",
        "components": {"c1": {"name": "P1"}},
        "adaptive_processes": [{"process_id": "p1", "instance_id": "c1", "process_type": "Iterative"}]
    },
    "dataspace": {
        "source_file": test_csv,
        "quality_objectives": [ {"name": "y", "metric": "unit"} ],
        "policy_identification": {
            "from": "column", 
            "policies": {}
        }
    }
}

with open(test_json, 'w') as f: 
    json.dump(system_def, f)

loader = GenericDataLoader()
sys_def = loader.load_system_definition(test_json)
print(f"Loaded System: {sys_def.system.name}")
print(f"Adaptive Processes: {[p.process_id for p in sys_def.system.adaptive_processes]}")

# Cleanup
os.remove(test_json)
os.remove(test_csv)

Registered parameter: replicas, Type: ParameterType.LEVER
Loaded System: ManualTest
Adaptive Processes: ['p1']


Phase 4

In [1]:
from adept.core.coordinator import ArchSpaceCore
import pandas as pd

# 1. Initialize the coordinator
core = ArchSpaceCore()
# 1. Initialize the coordinator
core = ArchSpaceCore()

# 2. Create sample continuous data
df = pd.DataFrame({
   'L1': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6], # Levers
   'Q1': [10, 20, 30, 40, 50, 60]  # Quality Outcome
 })

# 3. Verify formal Discretization Schemes
# We map Q1 to ['bad', 'good']
labels = {'Q1': ['bad', 'good']}
discrete_df, schemes = core.discretize(df, n_bins=2, all_labels=labels)

print("--- Discretization Schemes ---")
for scheme in schemes:
    print(f"Objective: {scheme.objective_name}")
    for b in scheme.bins:
        print(f"  Label: {b.label} -> Range: [{b.min_value:.2f}, {b.max_value:.2f}]")

# 4. Verify Discretization-based Scenario Discovery
# We want to find which L1 values lead to the 'good' bin
target_spec = {
    'paradigm': 'discretization',
    'target_bin': 'good'
}

# Run discovery targeting the 'good' categorical bin
box, limits, alg = core.discover_scenarios(
    df, 
    outcome='Q1', 
    target_spec=target_spec, 
    discrete_outcomes_df=discrete_df
)

print("\n--- Discovered Scenario for 'good' outcomes ---")
print(f"Parameter Limits: {limits}")

--- Discretization Schemes ---
Objective: L1
  Label: (0.0, 0.35] -> Range: [0.00, 0.35]
  Label: (0.35, 0.7] -> Range: [0.35, 0.70]
Objective: Q1
  Label: bad -> Range: [9.90, 35.00]
  Label: good -> Range: [35.00, 60.10]
Instances satisfying property: Q1
False    3
True     3
Name: count, dtype: int64
Total instances: (6, 2)
Running PRIM ... rhodium
2 possible boxes

--- Discovered Scenario for 'good' outcomes ---
Parameter Limits: {'L1': {'min': 0.35, 'max': 0.6}}


/Users/adiazpace/opt/anaconda3/envs/perfmodels/lib/python3.11/site-packages/prim/prim_box.py:415: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.peeling_trajectory = pd.concat([self.peeling_trajectory, pd.DataFrame([stats])],
/Users/adiazpace/opt/anaconda3/envs/perfmodels/lib/python3.11/site-packages/prim/prim_box.py:415: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.peeling_trajectory = pd.concat([self.peeling_trajectory, pd.DataFrame([stats])],
/Users/adiazpace/opt/anaconda3/envs/perf

In [2]:
schemes

[DiscretizationScheme(objective_name='L1', bins=[QualityBin(label='(0.0, 0.35]', min_value=0.0, max_value=0.35), QualityBin(label='(0.35, 0.7]', min_value=0.35, max_value=0.7)], method='equal_width'),
 DiscretizationScheme(objective_name='Q1', bins=[QualityBin(label='bad', min_value=9.9, max_value=35.0), QualityBin(label='good', min_value=35.0, max_value=60.1)], method='equal_width')]